<a href="https://colab.research.google.com/github/quiquefluque/Python-Finace/blob/main/Task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

#1.Carga y limpieza de datos
path="/content/drive/MyDrive/datasheet/Task 3 and 4_Loan_Data.csv"
datos=pd.read_csv(path)
df=pd.DataFrame(datos)

# 2.Divido datos
X = df.drop(columns=['customer_id', 'default'])
y = df['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3.Entreno modelo con regresión
model_log = LogisticRegression(max_iter=1000)#cerebro que estudia la relacion de datos,con polinomio de grado 1,otros cerebros son random tree, random forest
model_log.fit(X_train, y_train)#estudia la relacion entre preguntaspractica y respuestas practica

# 3. Entrenar un Random Forest
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

# 4. Prueba en grupo de pruebas regresion
y_pred_proba = model_log.predict_proba(X_test)[:, 1] #da como resultado una matriz de dos columnas, primera prob de pagador, la segunda prob de default,:de todas las filas,1 dame solo la segunda col
auc = roc_auc_score(y_test,y_pred_proba)#contrasta entre predicho y real

# 4. Prueba en grupo de pruebas forest
y_pred_proba_rf=model_rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test,y_pred_proba_rf)

#Resultados prints
print(f"AUC de Regresión Logística: {auc:.4f}")
print(f"AUC de Random Forest: {rf_auc:.4f}")

#Definir función
def calcular_perdida_esperada(datos_cliente, metodo='regresion', recovery_rate=0.10):
    """
    Función que predice el riesgo y la pérdida de un cliente.
    metodo: 'regresion' o 'forest'
    """
    # 1. Elegimos qué cerebro usar
    if metodo == 'regresion':
        prediccion_pd = model_log.predict_proba(datos_cliente)[0, 1]
    else:
        prediccion_pd = model_rf.predict_proba(datos_cliente)[0, 1]

    # 2. Datos del préstamo para el cálculo financiero
    # LGD = 1 - Tasa de recuperación (0.90 si recuperamos el 10%)
    lgd = 1 - recovery_rate
    # EAD = El monto(cantidad) que el cliente debe actualmente
    ead = datos_cliente['loan_amt_outstanding'].iloc[0]

    # 3. Fórmula: EL = PD * LGD * EAD
    pérdida_esperada = prediccion_pd * lgd * ead

    return prediccion_pd, pérdida_esperada #dos valores de salida de la funcion

#Prueba de funcion

# Tomamos un cliente cualquiera del grupo de pruebas (el primero, por ejemplo)
cliente_ejemplo = X_test.iloc[[0]]

# Calculamos con Regresión Logística
pd_log, el_log = calcular_perdida_esperada(cliente_ejemplo, metodo='regresion')

# Calculamos con Random Forest
pd_rf, el_rf = calcular_perdida_esperada(cliente_ejemplo, metodo='forest')

print(f"--- RESULTADOS CLIENTE ID: {y_test.index[0]} ---")
print(f"Probabilidad Default (Regresión): {pd_log:.2%}")
print(f"Probabilidad Default (Random Forest): {pd_rf:.2%}")
print(f"Pérdida Esperada Final: ${el_log:.2f}")





AUC de Regresión Logística: 1.0000
AUC de Random Forest: 0.9997
--- RESULTADOS CLIENTE ID: 6252 ---
Probabilidad Default (Regresión): 0.00%
Probabilidad Default (Random Forest): 0.00%
Pérdida Esperada Final: $0.00
